# Simulasi Turing Machine untuk Validasi Kata Sandi
**Mata Kuliah:** Teori Bahasa dan Automata  
**Nama:** Bima Satria  
**NIM:** 24343052  

Notebook ini adalah simulator edukatif berdasarkan studi kasus Makalah dan PPT Kelompok 11. Kode memperlihatkan tape, posisi head, state, penulisan simbol, gerak head, dan keputusan akhir. Ini bukan library eksternal atau implementasi produksi.

**Aturan validasi:** panjang minimal 8 karakter, minimal satu huruf kapital, satu huruf kecil, satu angka, dan satu simbol khusus. Materi Kelompok 11 menyimpulkan bahwa FA sudah cukup; PDA dan TM mampu menyelesaikan kasus ini, tetapi fitur tambahannya tidak diperlukan.

## 1. Representasi model
State direpresentasikan sebagai tuple `(jumlah_karakter, kapital, kecil, angka, simbol)`. Penghitung panjang dibatasi sampai 8 karena yang diperiksa hanya apakah panjang minimal telah tercapai. Tape berisi input diikuti simbol blank `□`. Head membaca dari kiri ke kanan, menulis kembali simbol yang sama, lalu bergerak ke kanan.

In [1]:
BLANK = "□"
ACCEPT = "q_accept"
REJECT = "q_reject"

def initial_state():
    return (0, False, False, False, False)

def classify_character(ch):
    if ch.isupper():
        return "kapital"
    if ch.islower():
        return "kecil"
    if ch.isdigit():
        return "angka"
    if not ch.isalnum():
        return "simbol"
    return "lainnya"

def update_state(state, ch):
    count, upper, lower, digit, special = state
    count = min(count + 1, 8)
    category = classify_character(ch)

    if category == "kapital":
        upper = True
    elif category == "kecil":
        lower = True
    elif category == "angka":
        digit = True
    elif category == "simbol":
        special = True

    return (count, upper, lower, digit, special)

def is_valid_state(state):
    count, upper, lower, digit, special = state
    return count >= 8 and upper and lower and digit and special

## 2. Fungsi simulasi Turing Machine

In [2]:
def simulate_tm(password, show_trace=True):
    tape = list(password) + [BLANK]
    head = 0
    state = initial_state()
    trace = []

    while True:
        symbol = tape[head]

        if symbol == BLANK:
            final_state = ACCEPT if is_valid_state(state) else REJECT
            trace.append((len(trace), final_state, head, BLANK, BLANK, "-", state))
            break

        state = update_state(state, symbol)
        tape[head] = symbol          # menulis kembali simbol yang sama
        trace.append((len(trace), "q", head, symbol, symbol, "R", state))
        head += 1                    # head bergerak ke kanan

    result = final_state == ACCEPT

    if show_trace:
        print("Kata sandi :", password)
        print("Hasil      :", "DITERIMA" if result else "DITOLAK")
        print("State akhir:", final_state)
        print("\nJejak komputasi:")
        for step, st, pos, read, written, move, indicators in trace:
            print(f"Langkah {step}: state={st}, head={pos}, baca={read!r}, "
                  f"tulis={written!r}, gerak={move}, indikator={indicators}")

    return result, trace

## 3. Pengujian contoh dari makalah

In [3]:
test_cases = [
    ("Belajar123!", True),
    ("belajar123!", False),
    ("BELAJAR123!", False),
    ("Belajar!!!!", False),
    ("Bela1!", False),
]

for password, expected in test_cases:
    result, _ = simulate_tm(password, show_trace=False)
    status = "DITERIMA" if result else "DITOLAK"
    print(f"{password:<15} -> {status:<10} | sesuai contoh: {result == expected}")

Belajar123!     -> DITERIMA   | sesuai contoh: True
belajar123!     -> DITOLAK    | sesuai contoh: True
BELAJAR123!     -> DITOLAK    | sesuai contoh: True
Belajar!!!!     -> DITOLAK    | sesuai contoh: True
Bela1!          -> DITOLAK    | sesuai contoh: True


## 4. Penelusuran input Belajar123!

In [4]:
hasil, jejak = simulate_tm("Belajar123!", show_trace=True)
print("\nInput diterima:", hasil)

Kata sandi : Belajar123!
Hasil      : DITERIMA
State akhir: q_accept

Jejak komputasi:
Langkah 0: state=q, head=0, baca='B', tulis='B', gerak=R, indikator=(1, True, False, False, False)
Langkah 1: state=q, head=1, baca='e', tulis='e', gerak=R, indikator=(2, True, True, False, False)
Langkah 2: state=q, head=2, baca='l', tulis='l', gerak=R, indikator=(3, True, True, False, False)
Langkah 3: state=q, head=3, baca='a', tulis='a', gerak=R, indikator=(4, True, True, False, False)
Langkah 4: state=q, head=4, baca='j', tulis='j', gerak=R, indikator=(5, True, True, False, False)
Langkah 5: state=q, head=5, baca='a', tulis='a', gerak=R, indikator=(6, True, True, False, False)
Langkah 6: state=q, head=6, baca='r', tulis='r', gerak=R, indikator=(7, True, True, False, False)
Langkah 7: state=q, head=7, baca='1', tulis='1', gerak=R, indikator=(8, True, True, True, False)
Langkah 8: state=q, head=8, baca='2', tulis='2', gerak=R, indikator=(8, True, True, True, False)
Langkah 9: state=q, head=9, baca

## 5. Pemetaan elemen teori ke kode

| Elemen teori | Representasi |
|---|---|
| Tape | `tape` pada `simulate_tm()` |
| Read/Write Head | Variabel `head` |
| State Register | Tuple `state` |
| Alfabet input | Karakter kata sandi |
| Blank | Konstanta `BLANK` |
| Fungsi transisi | `update_state()` dan loop simulasi |
| Write | `tape[head] = symbol` |
| Gerak kanan | `head += 1` |
| Accept / Reject | `q_accept` / `q_reject` |

## 6. Analisis dan batasan
- TM membaca input pada tape, memperbarui state, menulis kembali simbol yang sama, dan bergerak ke kanan sampai blank.
- Semua informasi dapat disimpan sebagai status berhingga. Karena itu FA sudah cukup dan merupakan model paling sederhana untuk kasus ini.
- PDA juga mampu menyelesaikan kasus, tetapi stack tidak diperlukan.
- Simulator ini memodelkan satu variasi TM deterministik satu pita dengan gerak head ke kanan. Ini bukan implementasi lengkap semua variasi TM.
- Simbol khusus disederhanakan sebagai karakter yang bukan huruf dan bukan angka. Pada sistem nyata, karakter yang diizinkan perlu ditentukan secara eksplisit.